In [1]:
from sentence_transformers import SentenceTransformer

In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [8]:
#Initialize the model
model=SentenceTransformer('all-MiniLM-L6-v2')

#simple text
text="""
The old clock in the hallway ticked louder than usual that night.
A stray cat followed me all the way to the bookstore.
The rain painted silver streaks across the windowpane.
She found a crumpled letter hidden inside the drawer.
The mountain air carried a silence that felt endless.

"""

#Step1:Splitting into sentences
sentences=[s.strip() for s in text.split("\n") if s.strip()]

In [9]:
#step 2:Embedding of each senteces
embeddings=model.encode(sentences)

In [10]:
threshold=0.7
chunks=[]
current_chunk=[sentences[0]]

In [12]:
#Step-4Semantic grouping based on threshold
for i in range(1,len(sentences)):
    sim=cosine_similarity(
        [embeddings[i-1]],[embeddings[i]]
    )[0][0]
    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append("".join(current_chunk))
        current_chunk=[sentences[i]]

#Append the last chunk
chunks.append("".join(current_chunk))
 

In [13]:
#Output of the above chunks
print("Semantic chunks")
for idx,chunk in enumerate(chunks):
    print(f"\n chunk {idx+1}:\n{chunk}")

Semantic chunks

 chunk 1:
The old clock in the hallway ticked louder than usual that night.

 chunk 2:
A stray cat followed me all the way to the bookstore.

 chunk 3:
The rain painted silver streaks across the windowpane.

 chunk 4:
She found a crumpled letter hidden inside the drawer.

 chunk 5:
The mountain air carried a silence that felt endless.


In [14]:
#Rag pipeline Modular Coding
import os

In [15]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [9]:
#Custom semantic chunker with threshold
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class ThresholdSemanticChunker:
    def __init__(self,model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model=SentenceTransformer(
            model_name
        )
        self.threshold=threshold

    def split(self,text:str):
        sentences=[s.strip() for s in text.split("\n") if s.strip()]
        embeddings=self.model.encode(sentences)
        chunks=[]
        current_chunk=[sentences[0]]

        for i in range(1,len(sentences)):
            sim=cosine_similarity([embeddings[i-1]],[embeddings[i]])[0][0]
            if sim>=self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(".".join(current_chunk)+".")
                current_chunk=[sentences[i]]
        chunks.append(".".join(current_chunk)+".")
        return chunks

    def split_documents(self, docs):
        result = []
        for doc in docs:
            chunks = self.split(doc.page_content)
            for chunk in chunks:
                result.append(Document(page_content=chunk, metadata=doc.metadata))
        return result

In [10]:
#Sample text
sample_text="""
Langchain is a framework for building applciation with LLM'set
Langchain provides modular abstractions to combine LLMS with tools like OPENAI and pinecone
You can create chains,agents,memory,and retriever
The Eiffel tower si located in Paris.
France is a popular tourist destination
"""
doc=Document(page_content=sample_text)
doc

Document(metadata={}, page_content="\nLangchain is a framework for building applciation with LLM'set\nLangchain provides modular abstractions to combine LLMS with tools like OPENAI and pinecone\nYou can create chains,agents,memory,and retriever\nThe Eiffel tower si located in Paris.\nFrance is a popular tourist destination\n")

In [32]:
#CHunking

chunker=ThresholdSemanticChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

[Document(metadata={}, page_content="Langchain is a framework for building applciation with LLM'set.Langchain provides modular abstractions to combine LLMS with tools like OPENAI and pinecone."),
 Document(metadata={}, page_content='You can create chains,agents,memory,and retriever.'),
 Document(metadata={}, page_content='The Eiffel tower si located in Paris..'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [36]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2'
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [45]:
#Vector store
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
retrieve=vectorstore.as_retriever()

In [46]:
print(type(embeddings))
print(embeddings)

<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [47]:
#Prompt Template
#5-The prompt template over here is
from langchain_core.prompts import PromptTemplate
template="""Answer the question based on the follwing context:
{context}
Question:{question}
"""
prompt=PromptTemplate.from_template(template)

In [69]:
#LLM here is
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.4)

In [70]:
#LCEL Chain with RETRIEVAL
from langchain_core.runnables import RunnableMap
from langchain_core.output_parsers import StrOutputParser
rag_chain=(
    RunnableMap(
        {
            "context":lambda x: retrieve.invoke(x['question']),
            "question":lambda x:x['question'],

        }
    )
    | prompt
    | llm
    | StrOutputParser()
)
query={"question":"What is LangChain used for?"}
result=rag_chain.invoke(query)
print(result)

Langchain is a framework for building applications with Large Language Models (LLMs). It provides modular abstractions to combine LLMS with tools like OPENAI and pinecone.


In [71]:
#Semantic CHunking in the langchain

In [72]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

In [76]:
#Loading the documents
loader=TextLoader('test.txt')
docs=loader.load()

In [77]:
#Initialize the medbdings model
embeddings=HuggingFaceEmbeddings()
#Create the semantic chunker
chunker=SemanticChunker(embeddings)

In [78]:
##SPlit the documents
chunk=chunker.split_documents(docs)

In [79]:
##Result
for i,chunk in enumerate(chunk):
    print(f"\n chunk {i+1}:\n{chunk.page_content}")


 chunk 1:
The old clock in the hallway ticked louder than usual that night. A stray cat followed me all the way to the bookstore. The rain painted silver streaks across the windowpane. She found a crumpled letter hidden inside the drawer. The mountain air carried a silence that felt endless.

 chunk 2:



In [80]:
#Making everything from the start to the end

In [85]:
embeddings=HuggingFaceEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2'
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [86]:
#Semantic chunkers
chunkers=SemanticChunker(embeddings)

In [87]:
# Example text
text = """
LangChain makes it easier to build applications with LLMs.
It provides tools for chaining, memory, and retrieval.
Semantic chunking improves retrieval by splitting text into meaningful segments.
"""


In [88]:
#Splitting text into semantic chunks
chunks=chunker.split_text(text )

In [89]:
#Vector base storing of an chunk
vectorstore=FAISS.from_texts(chunks,embedding=embeddings)

In [90]:
retriever=vectorstore.as_retriever()

In [92]:
#Prompt template
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    "Answer the question based on context:\n{context}\n\nQuestion: {question}"
)


In [93]:
llm=ChatGroq(
    model="openai/gpt-oss-120b"
)

In [95]:
#Parser
parser=StrOutputParser()

In [99]:
#LCEL PIPELINE 
rag_chain=(
    RunnableMap(
        {
            "context":lambda x: retrieve.invoke(x['question']),
            "question":lambda x:x['question'],

        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [103]:
result=rag_chain.invoke({"question":"What is the  role of langchain in Rag based systems?"})
print(result)

LangChain acts as the **glue‑code framework** that ties together all the pieces of a Retrieval‑Augmented Generation (RAG) pipeline.  
In a RAG system you typically need to:

1. **Retrieve** relevant documents from a vector store (e.g., Pinecone).  
2. **Pass** those documents to a large language model (LLM) for generation.  
3. **Maintain** context or memory across calls, and possibly **orchestrate** multiple steps (chains, agents).

LangChain provides modular abstractions for each of these steps:

- **Retrievers** – ready‑made components that query vector databases or other knowledge bases.  
- **Chains** – pipelines that sequence a retriever, a prompt template, an LLM call, and post‑processing logic.  
- **Agents** – dynamic decision‑making wrappers that can call tools (including retrievers) based on the LLM’s output.  
- **Memory** – utilities to keep track of prior interactions so the LLM can produce coherent, context‑aware responses.

By offering these building blocks, LangChain l